#### 7.1 Loading the Libraries

The required libraries were imported for model explainability and feature importance analysis.

The analysis uses model-based feature importance and SHAP to understand the contribution of input features to ride-cost predictions.

In [1]:
# Importing the required libraries

import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

#### 7.2 Loading the Models and Prepared Data

The selected models and prepared test dataset were loaded from the saved project artifacts.

Using the saved models and processed data ensures that explainability analysis is performed on the same feature representation and test observations used during model evaluation.

In [2]:
# Loading the prepared test data

X_test = pd.read_csv(
    "../data/processed/X_test.csv"
)

y_test = pd.read_csv(
    "../data/processed/y_test.csv"
).squeeze()

In [3]:
# Loading the selected models

selected_model_files = {
    'Lasso Regression': 'lasso_regression.joblib',
    'Stacking Regression': 'stacking_regression.joblib',
    'Gradient Boosting Tuned': 'gradient_boosting_tuned.joblib'
}

selected_models = {}

for model_name, file_name in selected_model_files.items():

    selected_models[model_name] = joblib.load(
        f"../models/{file_name}"
    )

for model_name, model in selected_models.items():
    print(
        f"{model_name}: {type(model).__name__}"
    )

Lasso Regression: Lasso
Stacking Regression: StackingRegressor
Gradient Boosting Tuned: GradientBoostingRegressor


#### 7.3 Feature Importance — Lasso Regression

Lasso Regression coefficients were examined to understand the direction and relative magnitude of each feature's contribution to predicted historical ride costs.

Positive coefficients indicate that an increase in the corresponding feature is associated with higher predicted ride costs, while negative coefficients indicate an association with lower predicted ride costs, holding the other features constant.

In [4]:
# Extracting Lasso coefficients

lasso_model = selected_models['Lasso Regression']

lasso_coefficients = pd.DataFrame({
    'Feature': X_test.columns,
    'Coefficient': lasso_model.coef_
})

lasso_coefficients['Absolute Coefficient'] = (
    lasso_coefficients['Coefficient'].abs()
)

lasso_coefficients = lasso_coefficients.sort_values(
    by='Absolute Coefficient',
    ascending=False
).reset_index(drop=True)

lasso_coefficients

,Feature,Coefficient,Absolute Coefficient
0,num__Expected_Ride_Duration,1.718644e+02,1.718644e+02
1,cat__Vehicle_Type_Economy,-4.230088e+01,4.230088e+01
2,num__Rider_Driver_Ratio,-5.815363e+00,5.815363e+00
3,cat__Location_Category_Rural,3.508095e+00,3.508095e+00
4,num__Average_Ratings,3.372285e+00,3.372285e+00
5,num__Number_of_Drivers,3.277797e+00,3.277797e+00
6,cat__Time_of_Booking_Evening,-3.223079e+00,3.223079e+00
7,num__Number_of_Riders,3.035645e+00,3.035645e+00
8,cat__Customer_Loyalty_Status_Gold,-2.839208e+00,2.839208e+00
9,cat__Time_of_Booking_Morning,2.067211e+00,2.067211e+00


In [6]:
# Visualizing the most influential Lasso coefficients
import plotly.express as px

top_lasso_features = lasso_coefficients.head(10).sort_values(
    by='Coefficient'
)

fig = px.bar(
    top_lasso_features,
    x='Coefficient',
    y='Feature',
    orientation='h',
    title='Top 10 Lasso Regression Coefficients',
    labels={
        'Coefficient': 'Coefficient',
        'Feature': 'Feature'
    }
)

fig.show()

#### Observation on Lasso Coefficients

`Expected_Ride_Duration` has the largest absolute coefficient by a substantial margin, with a positive coefficient of approximately 171.86. `Vehicle_Type_Economy` has the next largest absolute coefficient at approximately -42.30, followed by `Rider_Driver_Ratio` at approximately -5.82.

Among the remaining features, the coefficients are considerably smaller in magnitude. Lasso also assigns coefficients of zero or effectively zero to `Location_Category_Suburban`, `Customer_Loyalty_Status_Regular`, and `Vehicle_Type_Premium`, indicating that these encoded features were excluded from the model's linear contribution after regularization.

#### Important Point

The coefficient magnitudes indicate that expected ride duration is the dominant feature in the Lasso model, while the remaining variables provide smaller incremental contributions. The zero coefficients also demonstrate how Lasso regularization can reduce the influence of less informative features within the processed feature space.

#### 7.4 SHAP Explainability — Gradient Boosting

SHAP (SHapley Additive exPlanations) was used to analyze how individual features contribute to the predictions of the tuned Gradient Boosting model.

SHAP values indicate both the magnitude and direction of a feature's contribution to individual predictions. Positive SHAP values push a prediction higher, while negative SHAP values push it lower relative to the model's expected prediction.

In [7]:
# Loading the tuned Gradient Boosting model

gradient_boosting_model = selected_models[
    'Gradient Boosting Tuned'
]

In [8]:
# Creating the SHAP explainer

explainer = shap.TreeExplainer(
    gradient_boosting_model
)

shap_values = explainer.shap_values(
    X_test
)

print("SHAP values shape:", shap_values.shape)

SHAP values shape: (200, 18)


In [11]:
# Calculating global SHAP feature importance

shap_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean Absolute SHAP': np.abs(shap_values).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    by='Mean Absolute SHAP',
    ascending=False
).reset_index(drop=True)

shap_importance

,Feature,Mean Absolute SHAP
0,num__Expected_Ride_Duration,150.369253
1,cat__Vehicle_Type_Premium,10.451068
2,cat__Vehicle_Type_Economy,5.932571
3,num__Rider_Driver_Ratio,3.821277
4,num__Number_of_Drivers,3.696677
5,num__Average_Ratings,3.572604
6,num__Number_of_Riders,1.187384
7,num__Number_of_Past_Rides,0.848567
8,cat__Location_Category_Rural,0.487641
9,cat__Customer_Loyalty_Status_Gold,0.337991


In [12]:
# Visualizing global SHAP feature importance

top_shap_features = shap_importance.head(10).sort_values(
    by='Mean Absolute SHAP'
)

fig = px.bar(
    top_shap_features,
    x='Mean Absolute SHAP',
    y='Feature',
    orientation='h',
    title='Top 10 Features by Mean Absolute SHAP Value',
    labels={
        'Mean Absolute SHAP': 'Mean Absolute SHAP Value',
        'Feature': 'Feature'
    }
)

fig.show()

#### Observation on SHAP Feature Importance

The SHAP analysis identifies `Expected_Ride_Duration` as the dominant feature in the tuned Gradient Boosting model, with a mean absolute SHAP value of approximately 150.37. The vehicle-type features follow, with `Vehicle_Type_Premium` and `Vehicle_Type_Economy` having mean absolute SHAP values of approximately 10.45 and 5.93, respectively.

`Rider_Driver_Ratio`, `Number_of_Drivers`, and `Average_Ratings` also contribute to model predictions, although their average SHAP magnitudes are considerably smaller than those of ride duration and vehicle type.

#### Important Point

The SHAP results indicate that the tuned Gradient Boosting model relies primarily on expected ride duration, while vehicle type and demand-supply characteristics provide additional predictive information. Several categorical features have zero or near-zero mean absolute SHAP values, indicating limited contribution to the model's predictions within the evaluated test observations.

In [13]:
# Visualizing global SHAP feature importance

top_shap_features = shap_importance.head(10).sort_values(
    by='Mean Absolute SHAP'
)

fig = px.bar(
    top_shap_features,
    x='Mean Absolute SHAP',
    y='Feature',
    orientation='h',
    title='Top 10 Features by Mean Absolute SHAP Value',
    labels={
        'Mean Absolute SHAP': 'Mean Absolute SHAP Value',
        'Feature': 'Feature'
    }
)

fig.show()

#### 7.5 SHAP Directional Analysis

The direction of SHAP contributions was analyzed for the most influential features in the tuned Gradient Boosting model.

While mean absolute SHAP values measure the magnitude of feature influence, the signed SHAP values indicate whether a feature contributes toward increasing or decreasing the predicted historical ride cost.

In [14]:
# Calculating directional SHAP statistics

top_features = shap_importance.head(10)['Feature']

shap_directional = pd.DataFrame({
    'Feature': top_features,
    'Mean SHAP': [
        shap_values[:, X_test.columns.get_loc(feature)].mean()
        for feature in top_features
    ],
    'Mean Absolute SHAP': [
        shap_values[:, X_test.columns.get_loc(feature)].__abs__().mean()
        for feature in top_features
    ]
})

shap_directional = shap_directional.sort_values(
    by='Mean Absolute SHAP',
    ascending=False
).reset_index(drop=True)

shap_directional

,Feature,Mean SHAP,Mean Absolute SHAP
0,num__Expected_Ride_Duration,13.311952,150.369253
1,cat__Vehicle_Type_Premium,-0.139631,10.451068
2,cat__Vehicle_Type_Economy,-0.100626,5.932571
3,num__Rider_Driver_Ratio,0.122207,3.821277
4,num__Number_of_Drivers,-0.269683,3.696677
5,num__Average_Ratings,0.509903,3.572604
6,num__Number_of_Riders,-0.130501,1.187384
7,num__Number_of_Past_Rides,0.119353,0.848567
8,cat__Location_Category_Rural,0.070854,0.487641
9,cat__Customer_Loyalty_Status_Gold,-0.018628,0.337991


In [15]:
# Calculating positive and negative SHAP contributions

directional_summary = []

for feature in top_features:

    feature_index = X_test.columns.get_loc(feature)
    feature_shap = shap_values[:, feature_index]

    directional_summary.append({
        'Feature': feature,
        'Mean SHAP': feature_shap.mean(),
        'Positive Contribution (%)': (
            (feature_shap > 0).sum() / len(feature_shap)
        ) * 100,
        'Negative Contribution (%)': (
            (feature_shap < 0).sum() / len(feature_shap)
        ) * 100
    })

directional_summary = pd.DataFrame(
    directional_summary
)

directional_summary

,Feature,Mean SHAP,Positive Contribution (%),Negative Contribution (%)
0,num__Expected_Ride_Duration,13.311952,56.0,44.0
1,cat__Vehicle_Type_Premium,-0.139631,51.0,49.0
2,cat__Vehicle_Type_Economy,-0.100626,51.0,49.0
3,num__Rider_Driver_Ratio,0.122207,71.0,29.0
4,num__Number_of_Drivers,-0.269683,53.0,47.0
5,num__Average_Ratings,0.509903,54.5,45.5
6,num__Number_of_Riders,-0.130501,26.5,73.5
7,num__Number_of_Past_Rides,0.119353,68.0,32.0
8,cat__Location_Category_Rural,0.070854,38.0,62.0
9,cat__Customer_Loyalty_Status_Gold,-0.018628,71.0,29.0


#### Observation on SHAP Direction

The directional SHAP analysis shows that `Expected_Ride_Duration` has the largest average contribution, with a positive mean SHAP value of approximately 13.31 and positive contributions in 56% of the test observations. `Number_of_Riders` shows a negative mean SHAP value of approximately -0.13, with negative contributions in 73.5% of observations.

`Rider_Driver_Ratio` and `Number_of_Past_Rides` generally contribute positively, with positive contributions in 71% and 68% of observations respectively. In contrast, `Location_Category_Rural` and `Number_of_Drivers` show predominantly negative contributions, occurring in 62% and 47% of observations respectively.


#### Important Point

The SHAP analysis shows that influential features can affect individual predictions in both directions. Although `Expected_Ride_Duration` is the dominant feature by magnitude, its contribution is not uniformly positive across all observations. This highlights that the Gradient Boosting model uses feature values in combination rather than applying a fixed directional effect to every ride.


#### 7.6 Stacking Regression Component Analysis

The Stacking Regression model combines predictions from multiple base estimators and uses a final Ridge Regression model to generate the final prediction.

The component structure was examined to understand how the ensemble combines the individual base models used during training.

In [16]:
# Loading the Stacking Regression model

stacking_model = selected_models['Stacking Regression']

print("Base Estimators:")
for name, model in stacking_model.named_estimators_.items():
    print(f"- {name}: {type(model).__name__}")

print("\nFinal Estimator:")
print(type(stacking_model.final_estimator_).__name__)

Base Estimators:
- ridge: Ridge
- gradient_boosting: GradientBoostingRegressor
- xgboost: XGBRegressor
- lightgbm: LGBMRegressor

Final Estimator:
Ridge


#### Observation on Stacking Regression Components

The Stacking Regression model combines four base estimators: Ridge Regression, Gradient Boosting Regression, XGBoost, and LightGBM. Their predictions are passed to a final Ridge Regression estimator, which generates the final ensemble prediction.

This structure combines linear and non-linear modeling approaches within a single ensemble, allowing the final estimator to integrate the predictions produced by the different base models.


#### Important Point

The Stacking Regression model does not rely on a single underlying prediction mechanism. Instead, it combines predictions from multiple base models through a final Ridge Regression layer. This provides a model-level explanation of how the ensemble is structured, while feature-level explanations are provided separately through the Lasso coefficients and SHAP analysis.


#### 7.7 Final Explainability Summary

The explainability analysis provided complementary perspectives on how the selected models generate ride-cost predictions.

Lasso Regression showed the direction and relative magnitude of linear feature contributions through its coefficients. The analysis identified `Expected_Ride_Duration` as the dominant feature, followed by vehicle type and the rider-to-driver ratio.

SHAP analysis of the tuned Gradient Boosting model provided a model-independent view of feature contributions at the individual prediction level. It again identified `Expected_Ride_Duration` as the most influential feature, while vehicle type, rider-to-driver ratio, number of drivers, and average ratings provided additional predictive information.

The Stacking Regression analysis showed how multiple modeling approaches are combined through Ridge Regression, Gradient Boosting, XGBoost, and LightGBM base estimators, followed by a final Ridge Regression layer.

Together, these analyses provide both feature-level and model-level explanations of the selected predictive approaches.

#### Important Point

The explainability results consistently identify `Expected_Ride_Duration` as the most influential predictive feature across the linear and non-linear analyses. However, the remaining features contribute differently depending on the modeling approach, demonstrating that model interpretation should consider both the underlying model structure and the feature contribution method used.

#### 7.7 Permutation Feature Importance

Permutation feature importance was used to measure how much the tuned Gradient Boosting model's predictive performance changes when individual features are randomly shuffled.

A feature with a larger decrease in model performance after shuffling contributes more to the model's predictive performance. This provides a complementary feature-importance perspective to the SHAP analysis performed earlier.

A lightweight configuration with three repetitions and single-threaded execution was used to keep the analysis computationally efficient.

In [17]:
from sklearn.inspection import permutation_importance

In [18]:
# Calculating permutation feature importance for the tuned Gradient Boosting model

gradient_boosting_model = selected_models[
    'Gradient Boosting Tuned'
]

permutation_result = permutation_importance(
    gradient_boosting_model,
    X_test,
    y_test,
    scoring='neg_mean_absolute_error',
    n_repeats=3,
    random_state=42,
    n_jobs=1
)

print("Permutation importance calculated successfully.")

Permutation importance calculated successfully.


In [19]:
# Creating the permutation importance table

permutation_importance_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': permutation_result.importances_mean,
    'Importance Std': permutation_result.importances_std
})

permutation_importance_df = permutation_importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

permutation_importance_df.head(15)

,Feature,Importance,Importance Std
0,num__Expected_Ride_Duration,159.333359,4.489847
1,cat__Vehicle_Type_Premium,3.536381,0.467489
2,cat__Vehicle_Type_Economy,1.366764,0.235897
3,num__Average_Ratings,0.368750,0.124514
4,num__Number_of_Drivers,0.183938,0.288649
5,num__Rider_Driver_Ratio,0.079219,0.335950
6,num__Number_of_Riders,0.057629,0.114535
7,cat__Location_Category_Suburban,0.051267,0.037617
8,cat__Time_of_Booking_Morning,0.036526,0.027333
9,cat__Location_Category_Rural,0.007715,0.057555


In [20]:
# Visualizing the top 10 features based on permutation importance

top_permutation_features = (
    permutation_importance_df
    .head(10)
    .sort_values(by='Importance')
)

fig = px.bar(
    top_permutation_features,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Top 10 Features by Permutation Importance',
    labels={
        'Importance': 'Permutation Importance',
        'Feature': 'Feature'
    }
)

fig.show()

In [21]:
permutation_importance_df.head(15)

,Feature,Importance,Importance Std
0,num__Expected_Ride_Duration,159.333359,4.489847
1,cat__Vehicle_Type_Premium,3.536381,0.467489
2,cat__Vehicle_Type_Economy,1.366764,0.235897
3,num__Average_Ratings,0.368750,0.124514
4,num__Number_of_Drivers,0.183938,0.288649
5,num__Rider_Driver_Ratio,0.079219,0.335950
6,num__Number_of_Riders,0.057629,0.114535
7,cat__Location_Category_Suburban,0.051267,0.037617
8,cat__Time_of_Booking_Morning,0.036526,0.027333
9,cat__Location_Category_Rural,0.007715,0.057555


#### Observation on Permutation Feature Importance

The permutation importance analysis identifies `Expected_Ride_Duration` as the dominant feature in the tuned Gradient Boosting model, with an importance value of approximately 159.33. The two vehicle-type features follow, with `Vehicle_Type_Premium` and `Vehicle_Type_Economy` showing importance values of approximately 3.54 and 1.37 respectively.

`Average_Ratings`, `Number_of_Drivers`, and `Rider_Driver_Ratio` provide smaller contributions, while several categorical features show very low or zero permutation importance.

#### Important Point

The permutation importance results are broadly consistent with the SHAP analysis, which also identified `Expected_Ride_Duration` and vehicle type as the most influential features. This consistency across two different explainability methods strengthens the interpretation that ride duration is the primary predictive signal, with vehicle type and other operational variables providing additional information.